# Cecelia — Populations (`pop_df`)

How to pull populations out of an image with the unified `pop_df` accessor.

Scenario (image `KDIeEm`): **three segmentations** `A`, `B`, `C`. Each was gated
individually into a `qc` population, then tracked individually. We want:

- the gated cells of each segmentation (`flow`),
- all three pooled into one table,
- the **tracked** cells of each (`live` = the `qc` gate ∩ `track_id > 0`).

`pop_df` reads the H5AD via the `label_props` chain, evaluates gates in-process, and
pools across segmentations — see `docs/POPULATION.md`.

## Setup

In [ ]:
import Pkg; Pkg.activate(joinpath(@__DIR__, "..", "app"))

include("../app/src/Cecelia.jl")
using .Cecelia
using DataFrames

init_cecelia!()   # reads ~/cecelia-pineapple/dev/config.toml

## Load the image

`init_object(projectUID, imageUID)` dispatches on the stored class and returns a `CciaImage`.
The `label_props` dict tells us which segmentations exist (its keys are the value_names; `_active`
is the default one used when no `value_name` is passed).

In [ ]:
pID    = "NRUBxU"
imgUID = "KDIeEm"

img = init_object(pID, imgUID)
@show img.name
@show versioned_keys(img.label_props)   # the segmentations: ["A", "B", "C"]
img.label_props

## What's gated on each segmentation?

Each segmentation has its own gating sidecar `gating/{value_name}.json`. Load the map and list
its population paths — here each one has a single `qc` gate at the root (path `/qc`).

In [ ]:
for vn in versioned_keys(img.label_props)
    m = load_pop_map(img; value_name=vn, pop_type="flow")
    println("$vn  pop_type=$(m.pop_type)  pops=$(pop_paths(m))")
end

## A little helper

Most of the time we just want to see *how many* cells landed in each `(value_name, pop)`.

In [ ]:
summarise(df) = isempty(df) ? DataFrame() :
    sort!(combine(groupby(df, [:value_name, :pop]), nrow => :n), [:value_name, :pop])

## 1. Flow populations — one segmentation at a time

A **leading-slash** path (`"/qc"`) is resolved *within* the segmentation named by `value_name`.
`pop_cols` are read from the H5AD; intensity columns come back under their channel names by
default. Membership is computed by evaluating the gate in-process.

In [ ]:
qc_A = pop_df(img, "flow", ["/qc"]; value_name="A", pop_cols=["volume_mesh", "surface_to_volume"])
first(qc_A, 5)

In [ ]:
# the same call for B and C
qc_B = pop_df(img, "flow", ["/qc"]; value_name="B")
qc_C = pop_df(img, "flow", ["/qc"]; value_name="C")
(A = nrow(qc_A), B = nrow(qc_B), C = nrow(qc_C))

## 2. Pool all three segmentations in one call

A **prefixed** path (`"A/qc"`) names its own value_name, so one call can pull the same
population from several segmentations and `vcat` them into a single table (with a `value_name`
column). The pooled row count equals the sum of the per-segmentation counts above.

*(Note: you cannot mix value_names with leading-slash paths — `["/qc"]` is always relative to
the single `value_name` argument. Use the prefixed form to span segmentations.)*

In [ ]:
qc_all = pop_df(img, "flow", ["A/qc", "B/qc", "C/qc"]; pop_cols=["volume_mesh"])
@show nrow(qc_all)
summarise(qc_all)

## 3. Tracked (`live`) populations — the `_tracked` derived filter

`_tracked` is a **derived filtered population**, not a stored gate: it is the parent `qc` gate
intersected with `track_id > 0` (the `track_id` obs column written by the tracking task).
For `pop_type="live"`, `pop_df` loads the `flow` gates and layers this filter on at read time —
there is no `live` gating file. value_names are resolved automatically from the path prefix.

Derived populations live in a **reserved namespace: leaf names beginning with `_`** (so a
hand-drawn gate can never be named `_tracked` and shadow it). Population clustering will register
its derived pops the same way.

In [ ]:
tracked = pop_df(img, "live", ["A/qc/_tracked", "B/qc/_tracked", "C/qc/_tracked"]; pop_cols=["track_id"])
@show nrow(tracked)
@show all(tracked.track_id .> 0)   # every cell is tracked
summarise(tracked)

Tracked cells are a subset of the gated cells — the same `qc` cells, minus the ones that
never received a track.

In [ ]:
DataFrame(
    value_name = ["A", "B", "C"],
    qc         = [nrow(qc_A), nrow(qc_B), nrow(qc_C)],
    tracked    = [nrow(tracked[tracked.value_name .== vn, :]) for vn in ["A", "B", "C"]],
)

## Extras

- `value_name=nothing` (the default) uses the **active** segmentation (`_active`).
- `drop_na=true` drops cells that are NA/NaN in any requested `pop_col`.
- `raw_channel_names=true` keeps the raw `{measure}_intensity_{i}` names instead of channel names.
- Results are cached per image and **auto-invalidate** when the gating map or H5AD changes on
  disk (re-gate / re-track) — `flush_cache=true` forces a recompute.

In [ ]:
# active segmentation (no value_name) + drop cells with NaN track_id
active = pop_df(img, "flow", ["/qc"]; pop_cols=["track_id"], drop_na=true)
@show versioned_keys(img.label_props)
@show img.label_props["_active"]
@show nrow(active)
first(active, 5)